In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 64. Week 44 — Portfolio decision under estimation and cost uncertainty

## 学習目標


- Treasury yield-change covarianceをcurve exposure allocationへ接続する。
- shrinkage、turnover penalty、risk budget、scenario sensitivityを同時に診断する。
- weightsをyield exposureと定義し、cash security portfolioやrealized PnLと呼ばない。


## 前提知識


- B1のcurve tenors、B4のquadratic allocation、B5–B7のchronological split
- covariance、ridge/shrinkage、turnover penaltyの基礎

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 64


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Exposure allocation contract

ここでのweightは5つのTreasury par-yield changeへの線形exposureである。目的関数の単位はbpとbp²であり、債券notional、duration、cash return、execution PnLを含まない。

$$
\max_w\ \mu^\top w-\frac{\gamma}{2}w^\top\Sigma w-\tau\lVert w-w_{prev}\rVert_2^2,\qquad \mathbf{1}^\top w=0
$$

In [4]:
changes_bp = curve_changes_bp
train_end = int(0.70 * changes_bp.shape[0])
training_changes = changes_bp[:train_end]
expected_change = training_changes.mean(axis=0)
sample_covariance = np.cov(training_changes, rowvar=False, ddof=1)
shrinkage_intensity = 0.25
diagonal_target = np.diag(np.diag(sample_covariance))
covariance = (1.0 - shrinkage_intensity) * sample_covariance + shrinkage_intensity * diagonal_target
allocation = qt.mean_variance_allocation(
    expected_change,
    covariance,
    risk_aversion=5.0,
    previous_weights=np.zeros(curve_yields.shape[1]),
    turnover_penalty=0.50,
    net_exposure=0.0,
)
assert allocation.budget_residual < 1e-10
allocation_table = pd.DataFrame(
    {
        "tenor": qt.DEFAULT_TENORS,
        "expected_change_bp": expected_change,
        "exposure_weight": allocation.weights,
    }
)
display(allocation_table)
print("training rows:", training_changes.shape[0], "stationarity residual:", allocation.stationarity_residual)

,tenor,expected_change_bp,exposure_weight
0,3m,0.159044,0.000536
1,2y,0.150728,0.000882
2,5y,0.095634,-0.000126
3,10y,0.062890,-0.000469
4,30y,0.040541,-0.000824


training rows: 1924 stationarity residual: 2.7755575615628914e-17


In [5]:
sensitivity_rows = []
for penalty in (0.0, 0.10, 0.50, 2.0):
    result = qt.mean_variance_allocation(
        expected_change,
        covariance,
        risk_aversion=5.0,
        previous_weights=np.zeros(curve_yields.shape[1]),
        turnover_penalty=penalty,
        net_exposure=0.0,
    )
    for tenor, weight in zip(qt.DEFAULT_TENORS, result.weights, strict=True):
        sensitivity_rows.append({"turnover_penalty": penalty, "tenor": tenor, "weight": weight})
sensitivity_table = pd.DataFrame(sensitivity_rows)
fig = go.Figure()
for tenor in qt.DEFAULT_TENORS:
    subset = sensitivity_table[sensitivity_table["tenor"] == tenor]
    fig.add_scatter(x=subset["turnover_penalty"], y=subset["weight"], mode="lines+markers", name=tenor)
fig.update_layout(title="Exposure sensitivity to turnover penalty", xaxis_title="Turnover penalty", yaxis_title="Yield-change exposure weight", template="plotly_white")
fig.show()

## 2. 失敗モード

- 全期間の平均・共分散で過去のdecision originを汚染する。
- covariance shrinkageをvalidation後に選ぶ。
- weightをsecurity holding、PnL、hedge ratioと呼ぶ。
- turnover penaltyをspreadの推定値として扱う。

## 3. 段階別演習

### 基礎

1. net_exposure=0がcurve exposureで何を意味するか説明せよ。

### 標準

2. shrinkage intensityをtrainingだけで比較し、testへ再選択しない表を作れ。

### 研究

3. funding、capacity、crowdingを加えるために必要な観測と、現データで許されるscenarioを分けよ。

## 4. Exit Criteria

- [ ] covariance fitがtraining partitionだけである
- [ ] shrinkageとturnover sensitivityを保存した
- [ ] exposure unitをbp変化として明記した
- [ ] cash portfolio/PnL claimをしていない

## 5. 出典

- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)
- [U.S. Treasury Daily Treasury Par Yield Curve Rates](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve)

- [Fama and MacBeth (1973), Risk, Return, and Equilibrium](https://www.jstor.org/stable/1831028)
- [Hansen (1982), Large Sample Properties of GMM Estimators](https://doi.org/10.2307/1912775)
- [Duffie and Kan (1996), A Yield-Factor Model of Interest Rates](https://doi.org/10.1016/0304-405X(95)00881-6)
- [Boyd and Vandenberghe, Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/bv_cvxbook.pdf)